# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("data/ps3_gwas.phen", sep="\t", header=None)

# Clean and Align Phenotype Data

In [4]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS Loop

In [ ]:
# Set up output file
with open("gwas_results.tsv", "w") as out:
    out.write("CHR\tPOS\tBETA\tP\n")

    # Used for regression, calculated outside of loop for efficiency
    y0 = y - y.mean()
    df = len(y) - 2

    # SNP loop
    max_snps = 50
    count = 0
    for v in vcf:
        # Limit SNPs to process
        if count >= max_snps:
            break

        # Convert genotype to doage
        g = v.gt_types
        g = g.astype(float)
        g[g == 3] = np.nan

        # If no valid genotype data, continue to next SNP
        if np.isnan(g).all():
            continue

        # Replace missing with mean
        g = np.where(np.isnan(g), np.nanmean(g), g)

        # Find MAF
        maf = min(g.mean() / 2, 1 - g.mean() / 2)

        # Filter rare variants
        if maf < 0.01:
            continue

        # Regression
        g0 = g - g.mean()

        beta = (g0 @ y0) / (g0 @ g0)

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / df / (g0 @ g0))

        t = beta/se
        p = 2 * stats.t.sf(abs(t), df)

        # Write to file
        out.write(f"{v.CHROM}\t{v.POS}\t{beta}\t{p}\n")

        # Increment counter
        count += 1